# GRPO GSM8K training (Colab/Kaggle T4)

This notebook clones the repo, installs dependencies, and runs `train.py`.
Select **Runtime > Change runtime type > T4 GPU** first.

The full config trains Qwen2.5-0.5B-Instruct with LoRA, 8 generations per
question, for 200 optimizer steps. Expect a few hours on a free T4.

In [ ]:
!nvidia-smi

In [ ]:
import os

# Replace with your fork if you have not pushed this repo yet.
REPO_URL = "https://github.com/YOUR_USERNAME/grpo-small-reasoner.git"
REPO_DIR = "/content/grpo-small-reasoner"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print(os.getcwd())

In [ ]:
# Install project deps into Colab's Python. Tokens are read from Colab Secrets.
!pip install -q uv
!uv pip install --system -e .

In [ ]:
import os

try:
    from google.colab import userdata

    for key in ("WANDB_API_KEY", "HF_TOKEN"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            print(f"{key} not found in Colab secrets; skipping")
except ImportError:
    print("Not running on Colab; set WANDB_API_KEY / HF_TOKEN in the environment")

os.environ.setdefault("WANDB_PROJECT", "grpo-small-reasoner")

Optional edits before training:

- Change `model.name` to `Qwen/Qwen2.5-1.5B-Instruct` for the 1.5B option.
- Set `hub.push: true` and `hub.repo_id: your-name/grpo-qwen2.5-0.5b-gsm8k`.
- If the T4 runs out of memory, use `per_device_train_batch_size: 4` with
  `gradient_accumulation_steps: 2` (still 8 completions per generation batch).

In [ ]:
# Inspect or edit with: !sed -n '1,120p' configs/train_full.yaml
!python train.py --config configs/train_full.yaml

In [ ]:
# Download the adapter and metrics for Phase 4 evaluation/plots.
import shutil
from pathlib import Path

from google.colab import files

run_dir = Path("outputs/grpo-qwen2.5-0.5b")
archive = shutil.make_archive("/content/grpo_adapter", "zip", run_dir)
print(archive)
files.download(archive)